In [1]:
import csv
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================================
# GLOBAL DATAFRAMES
# ==========================================================

df_internal_incidents = None
df_sgh_aia = None
df_incident_report_2025 = None
df_incident_report_2026 = None

# ==========================================================
# PLACEHOLDER FOR REPORT BUTTON
# ==========================================================

run_report_button = None

# ==========================================================
# FILE INPUT WIDGETS
# ==========================================================

folder_input = widgets.Text(
    value="C:/Users/C28369/Documents/Monthly_Reports/AP Presentations/July/",
    description="Base Folder:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%")
)

file1_input = widgets.Text(
    value="Internal Incidents PG.csv",
    description="File 1 (CSV):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%")
)

file2_input = widgets.Text(
    value="SGH AIA.xlsx",
    description="File 2 (XLSX):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%")
)

file3_input = widgets.Text(
    value="SGH Weekly Incident Report Data - All Brands YTD 073125.csv",
    description="File 3 (2025 CSV):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%")
)

file4_input = widgets.Text(
    value="SGH Weekly Incident Report Data - All Brands YTD 073126.csv",
    description="File 4 (2026 CSV):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%")
)

load_button = widgets.Button(
    description="Load Data",
    button_style="success",
    layout=widgets.Layout(width="250px")
)

output_area = widgets.Output()

# ==========================================================
# LOAD DATA
# ==========================================================

def load_data_callback(b):

    global df_internal_incidents
    global df_sgh_aia
    global df_incident_report_2025
    global df_incident_report_2026
    global run_report_button

    with output_area:

        clear_output()

        try:

            print("Loading data...")

            base_folder = Path(folder_input.value.strip())

            df_internal_incidents = pd.read_csv(
                base_folder / file1_input.value.strip()
            )

            df_sgh_aia = pd.read_excel(
                base_folder / file2_input.value.strip(),
                sheet_name="Report 1"
            )

            df_incident_report_2025 = pd.read_csv(
                base_folder / file3_input.value.strip()
            )

            df_incident_report_2026 = pd.read_csv(
                base_folder / file4_input.value.strip()
            )

            print("✅ Data loaded successfully")
            print(f"Internal Incidents: {len(df_internal_incidents):,} rows")
            print(f"SGH AIA: {len(df_sgh_aia):,} rows")
            print(f"2025 Incident Report: {len(df_incident_report_2025):,} rows")
            print(f"2026 Incident Report: {len(df_incident_report_2026):,} rows")

            if run_report_button is not None:
                run_report_button.disabled = False

        except Exception as e:

            print(f"❌ Error loading data: {e}")

load_button.on_click(load_data_callback)

display(
    widgets.HTML("<h3>AP Presentation Data Loader</h3>"),
    folder_input,
    file1_input,
    file2_input,
    file3_input,
    file4_input,
    load_button,
    output_area
)


HTML(value='<h3>AP Presentation Data Loader</h3>')

Text(value='C:/Users/C28369/Documents/Monthly_Reports/AP Presentations/July/', description='Base Folder:', lay…

Text(value='Internal Incidents PG.csv', description='File 1 (CSV):', layout=Layout(width='80%'), style=TextSty…

Text(value='SGH AIA.xlsx', description='File 2 (XLSX):', layout=Layout(width='80%'), style=TextStyle(descripti…

Text(value='SGH Weekly Incident Report Data - All Brands YTD 073125.csv', description='File 3 (2025 CSV):', la…

Text(value='SGH Weekly Incident Report Data - All Brands YTD 073126.csv', description='File 4 (2026 CSV):', la…

Button(button_style='success', description='Load Data', layout=Layout(width='250px'), style=ButtonStyle())

Output()

In [28]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display

run_report_button = widgets.Button(
    description="Run External Incident Report",
    button_style="primary"
)

report_output = widgets.Output()

display(run_report_button)
display(report_output)


def run_report(btn=None):

    with report_output:

        report_output.clear_output()

        if (
            df_incident_report_2025 is None
            or df_incident_report_2026 is None
        ):
            display(HTML(
                "<h3 style='color:red;'>Please load data first.</h3>"
            ))
            return

        stores = sorted(
            pd.concat([
                df_incident_report_2025["Store Name"],
                df_incident_report_2026["Store Name"]
            ])
            .dropna()
            .unique()
            .tolist()
        )

        dropdown = widgets.SelectMultiple(
            options=stores,
            value=("Pearle Vision",)
            if "Pearle Vision" in stores
            else (stores[0],),
            description="Store(s):",
            rows=min(15, len(stores)),
            layout=widgets.Layout(
                width="450px",
                height="250px"
            )
        )

        output = widgets.Output()

        display(dropdown)
        display(output)

        def format_cost(x):

            if x == 0:
                return "$0"

            if x >= 1000:
                return f"${round(x/1000):.0f}k"

            return f"${x:,.0f}"

        def format_pct(curr, prev):

            if prev == 0 and curr > 0:
                return "100%"

            if prev == 0 and curr == 0:
                return "0%"

            return f"{round(((curr-prev)/prev)*100):.0f}%"

        def color_box(val):

            if val == "0%":
                color = "#fff3cd"      # yellow
            elif val.startswith("-"):
                color = "#d4edda"      # green
            else:
                color = "#f8d7da"      # red

            return (
                f"<div style='background:{color};"
                f"padding:4px;border-radius:4px;"
                f"margin:2px;font-weight:bold;'>"
                f"{val}</div>"
            )

        def build_report(change=None):

            with output:

                output.clear_output()

                selected_stores = list(dropdown.value)
                
                if len(selected_stores) == 0:
                    display(
                        HTML(
                            "<h4 style='color:red;'>Please select at least one store.</h4>"
                        )
                    )
                    return

                df25 = df_incident_report_2025[
                    df_incident_report_2025["Store Name"]
                    .isin(selected_stores)
                ].copy()

                df26 = df_incident_report_2026[
                    df_incident_report_2026["Store Name"]
                    .isin(selected_stores)
                ].copy()

                for df in [df25, df26]:

                    df["Total Stolen"] = (
                        df["Total Stolen"]
                        .astype(str)
                        .str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False)
                    )

                    df["Total Stolen"] = pd.to_numeric(
                        df["Total Stolen"],
                        errors="coerce"
                    ).fillna(0)

                categories = {
                    "Active Shooter": "Active Shooter",
                    "Burglary": "Burglary",
                    "External Theft/Shoplifting":
                        "External Theft/Shoplifting",
                    "Robbery":
                        "Robbery|Violent Crime/Threats"
                }

                rows = []

                for name, regex in categories.items():

                    a = df25[
                        df25["Case Type"]
                        .str.contains(regex, case=False, na=False)
                    ]

                    b = df26[
                        df26["Case Type"]
                        .str.contains(regex, case=False, na=False)
                    ]

                    raw_cost25 = a["Total Stolen"].sum()
                    raw_cost26 = b["Total Stolen"].sum()

                    display_cost25 = (
                        round(raw_cost25 / 1000) * 1000
                        if raw_cost25 >= 1000
                        else raw_cost25
                    )

                    display_cost26 = (
                        round(raw_cost26 / 1000) * 1000
                        if raw_cost26 >= 1000
                        else raw_cost26
                    )

                    rows.append({
                        "Incident Type": name,
                        "Count25": len(a),
                        "Cost25": display_cost25,
                        "Count26": len(b),
                        "Cost26": display_cost26
                    })

                summary = pd.DataFrame(rows)

                total = {
                    "Incident Type": "Total",
                    "Count25": int(summary["Count25"].sum()),
                    "Cost25": int(summary["Cost25"].sum()),
                    "Count26": int(summary["Count26"].sum()),
                    "Cost26": int(summary["Cost26"].sum())
                }

                if len(selected_stores) == 1:
                
                    report_title = (
                        f"{selected_stores[0]} - External Incidents and "
                        f"Total Reported Stolen 2025 vs 2026 (YTD)"
                    )
                
                else:
                
                    report_title = (
                        f"{len(selected_stores)} Selected Stores - "
                        f"External Incidents and Total Reported "
                        f"Stolen 2025 vs 2026 (YTD)"
                    )
                
                selected_store_text = ", ".join(selected_stores)
                
                html_rows = ""

                for _, r in summary.iterrows():

                    cnt_change = format_pct(
                        int(r["Count26"]),
                        int(r["Count25"])
                    )

                    cost_change = format_pct(
                        int(r["Cost26"]),
                        int(r["Cost25"])
                    )

                    html_rows += f"""
                    <tr>
                        <td>{r['Incident Type']}</td>

                        <td>
                            {int(r['Count25'])}<br>
                            {format_cost(r['Cost25'])}
                        </td>

                        <td>
                            {int(r['Count26'])}<br>
                            {format_cost(r['Cost26'])}
                        </td>

                        <td>
                            {color_box(cnt_change)}
                            {color_box(cost_change)}
                        </td>

                    </tr>
                    """

                total_cnt_change = format_pct(
                    total["Count26"],
                    total["Count25"]
                )

                total_cost_change = format_pct(
                    total["Cost26"],
                    total["Cost25"]
                )

                html_rows += f"""
                <tr style="
                    font-weight:bold;
                    border-top:2px solid black;
                ">

                    <td>Total</td>

                    <td>
                        {int(total['Count25'])}<br>
                        {format_cost(total['Cost25'])}
                    </td>

                    <td>
                        {int(total['Count26'])}<br>
                        {format_cost(total['Cost26'])}
                    </td>

                    <td>
                        {color_box(total_cnt_change)}
                        {color_box(total_cost_change)}
                    </td>

                </tr>
                """

                html = f"""
                <h3 style="text-align:center;">
                    {report_title}
                </h3>
                
                <div style="
                    text-align:center;
                    color:gray;
                    font-size:8pt;
                    margin-bottom:10px;
                ">
                    {selected_store_text}
                </div>

                <table
                    style="
                        width:80%;
                        margin:auto;
                        border-collapse:collapse;
                        font-family:Arial;
                    "
                    border="1"
                >

                    <tr style="
                        background:#6da97c;
                        color:white;
                    ">
                        <th>Incident Type</th>
                        <th>2025</th>
                        <th>2026</th>
                        <th>Change</th>
                    </tr>

                    {html_rows}

                </table>
                """

                display(HTML(html))

        dropdown.observe(build_report, names="value")

        build_report()

run_report_button.on_click(run_report)

Button(button_style='primary', description='Run External Incident Report', style=ButtonStyle())

Output()

In [37]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display

# ============================================================
# GROUP SELECTOR
# ============================================================

group_options = sorted(
    df_sgh_aia["Group"]
    .dropna()
    .unique()
    .tolist()
)

group_selector = widgets.SelectMultiple(
    options=group_options,
    value=(
        "Pearle Vision CN",
        "Pearle Vision US"
    ),
    description="Group(s):",
    rows=min(10, len(group_options)),
    layout=widgets.Layout(
        width="450px",
        height="200px"
    )
)

output = widgets.Output()

display(group_selector)
display(output)

# ============================================================
# REPORT FUNCTION
# ============================================================

def build_report(change=None):

    with output:

        output.clear_output()

        selected_groups = list(group_selector.value)

        if len(selected_groups) == 0:

            display(
                HTML(
                    """
                    <div style="
                        color:red;
                        font-family:Arial;
                        font-size:12pt;
                    ">
                        Please select at least one group.
                    </div>
                    """
                )
            )

            return

        # ----------------------------------------------------
        # FILTER GROUPS
        # ----------------------------------------------------

        df_filtered = df_sgh_aia[
            df_sgh_aia["Group"].isin(selected_groups)
        ].copy()

        # ----------------------------------------------------
        # KEEP REQUIRED COLUMNS
        # ----------------------------------------------------

        columns_to_keep = [
            "Store #",
            "Store Name",
            "City",
            "State",
            "Movement Quantity",
            "Movement Internal Cost Home"
        ]

        df_filtered = df_filtered[columns_to_keep].copy()

        # ----------------------------------------------------
        # CONVERT VALUES
        # ----------------------------------------------------

        df_filtered["Movement Quantity"] = (
            df_filtered["Movement Quantity"]
            .fillna(0)
            .astype(float)
            .abs()
        )

        # Convert Cost by multiplying by -1
        df_filtered["Movement Internal Cost Home"] = (
            df_filtered["Movement Internal Cost Home"]
            .fillna(0)
            .astype(float)
            * -1
        )

        # ----------------------------------------------------
        # AGGREGATE TO STORE LEVEL
        # ----------------------------------------------------

        df_pivot = df_filtered.pivot_table(
            index=[
                "Store #",
                "Store Name",
                "City",
                "State"
            ],
            values=[
                "Movement Quantity",
                "Movement Internal Cost Home"
            ],
            aggfunc="sum"
        ).reset_index()

        # ----------------------------------------------------
        # RENAME COLUMNS
        # ----------------------------------------------------

        df_pivot = df_pivot.rename(
            columns={
                "Store #": "Store",
                "Movement Quantity": "Units",
                "Movement Internal Cost Home": "Cost"
            }
        )

        # ----------------------------------------------------
        # SORT BY COST DESCENDING
        # ----------------------------------------------------

        df_final = df_pivot.sort_values(
            by="Cost",
            ascending=False
        )

        # ----------------------------------------------------
        # OUTPUT COLUMNS
        # ----------------------------------------------------

        desired_columns = [
            "Store",
            "Store Name",
            "City",
            "State",
            "Units",
            "Cost"
        ]

        df_final = df_final[desired_columns]

        # ----------------------------------------------------
        # FORMAT DISPLAY
        # ----------------------------------------------------

        df_display = df_final.copy()

        df_display["Units"] = (
            df_display["Units"]
            .round(0)
            .astype(int)
            .map("{:,}".format)
        )

        df_display["Cost"] = (
            df_display["Cost"]
            .round(0)
            .astype(int)
            .map("${:,.0f}".format)
        )

        # ----------------------------------------------------
        # DYNAMIC TITLE
        # ----------------------------------------------------

        if len(selected_groups) == 1:

            group_label = selected_groups[0]

        elif len(selected_groups) <= 3:

            group_label = ", ".join(selected_groups)

        else:

            group_label = (
                f"{len(selected_groups)} Groups Selected"
            )

        report_title = (
            "Top Stores by Adjusted Theft Units "
            "& Stat Cost YTD"
        )

        # ----------------------------------------------------
        # DISPLAY TITLE
        # ----------------------------------------------------

        display(
            HTML(
                f"""
                <h3 style="
                    text-align:center;
                    margin-bottom:5px;
                    font-family:Arial;
                ">
                    {report_title}
                </h3>

                <div style="
                    text-align:center;
                    color:gray;
                    font-size:8pt;
                    margin-bottom:15px;
                    font-family:Arial;
                ">
                    {group_label}
                </div>
                """
            )
        )

        # ----------------------------------------------------
        # DISPLAY TABLE (TOP 5, NO INDEX, CENTERED)
        # ----------------------------------------------------

        display_df = df_display.head(5)

        centered_html = f"""
        <div style="
            display:flex;
            justify-content:center;
            width:100%;
        ">
            {display_df.style.hide(axis='index').to_html()}
        </div>
        """

        display(HTML(centered_html))

# ============================================================
# EVENTS
# ============================================================

group_selector.observe(
    build_report,
    names="value"
)

# ============================================================
# INITIAL LOAD
# ============================================================

build_report()

SelectMultiple(description='Group(s):', index=(4, 6), layout=Layout(height='200px', width='450px'), options=('…

Output()

In [50]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display

# ============================================================
# STORE NAME SELECTOR (MULTI-SELECT)
# ============================================================

store_options = sorted(
    df_incident_report_2026["Store Name"]
    .dropna()
    .unique()
    .tolist()
)

store_selector = widgets.SelectMultiple(
    options=store_options,
    value=("Pearle Vision",)
    if "Pearle Vision" in store_options
    else (store_options[0],),
    description="Store(s):",
    rows=min(10, len(store_options)),
    layout=widgets.Layout(
        width="450px",
        height="200px"
    )
)

output = widgets.Output()

display(store_selector)
display(output)

# ============================================================
# REPORT FUNCTION
# ============================================================

def build_report(change=None):

    with output:

        output.clear_output()

        selected_stores = list(store_selector.value)

        if len(selected_stores) == 0:

            display(
                HTML(
                    """
                    <div style="
                        color:red;
                        font-family:Arial;
                        font-size:12pt;
                    ">
                        Please select at least one store.
                    </div>
                    """
                )
            )

            return

        # ----------------------------------------------------
        # FILTER STORE NAMES
        # ----------------------------------------------------

        df_filtered = df_incident_report_2026[
            df_incident_report_2026["Store Name"]
            .isin(selected_stores)
        ].copy()

        # ----------------------------------------------------
        # KEEP REQUIRED COLUMNS
        # ----------------------------------------------------

        columns_to_keep = [
            "Store No",
            "Location",
            "City",
            "State",
            "Total Units",
            "Total Stolen"
        ]

        df_filtered = df_filtered[columns_to_keep].copy()

        # ----------------------------------------------------
        # CLEAN TOTAL STOLEN
        # ----------------------------------------------------

        df_filtered["Total Stolen"] = (
            df_filtered["Total Stolen"]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
        )

        df_filtered["Total Stolen"] = pd.to_numeric(
            df_filtered["Total Stolen"],
            errors="coerce"
        ).fillna(0)

        # ----------------------------------------------------
        # ENSURE TOTAL STOLEN IS POSITIVE
        # ----------------------------------------------------

        df_filtered["Total Stolen"] = (
            df_filtered["Total Stolen"]
            .abs()
        )

        # ----------------------------------------------------
        # AGGREGATE TO STORE LEVEL
        # ----------------------------------------------------

        df_pivot = df_filtered.pivot_table(
            index=[
                "Store No",
                "Location",
                "City",
                "State"
            ],
            values=[
                "Total Units",
                "Total Stolen"
            ],
            aggfunc={
                "Total Units": "count",
                "Total Stolen": "sum"
            }
        ).reset_index()

        # ----------------------------------------------------
        # RENAME COLUMNS
        # ----------------------------------------------------

        df_pivot = df_pivot.rename(
            columns={
                "Store No": "Store",
                "Location": "Store Name",
                "Total Units": "Incidents"
            }
        )

        # ----------------------------------------------------
        # SORT BY TOTAL STOLEN DESCENDING
        # ----------------------------------------------------

        df_final = (
            df_pivot
            .sort_values(
                by="Total Stolen",
                ascending=False
            )
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # OUTPUT COLUMNS
        # ----------------------------------------------------

        desired_columns = [
            "Store",
            "Store Name",
            "City",
            "State",
            "Incidents",
            "Total Stolen"
        ]

        df_final = df_final[desired_columns]

        # ----------------------------------------------------
        # FORMAT DISPLAY
        # ----------------------------------------------------

        df_display = df_final.copy()

        df_display["Incidents"] = (
            df_display["Incidents"]
            .astype(int)
            .map("{:,}".format)
        )

        df_display["Total Stolen"] = (
            df_display["Total Stolen"]
            .round(0)
            .astype(int)
            .map("${:,}".format)
        )

        # ----------------------------------------------------
        # DYNAMIC TITLE
        # ----------------------------------------------------

        if len(selected_stores) == 1:

            store_label = selected_stores[0]

        elif len(selected_stores) <= 3:

            store_label = ", ".join(selected_stores)

        else:

            store_label = (
                f"{len(selected_stores)} Stores Selected"
            )

        # ----------------------------------------------------
        # TOP 5 ROWS
        # ----------------------------------------------------

        display_df = df_display.head(5)

        # ----------------------------------------------------
        # HTML TABLE
        # ----------------------------------------------------

        html_table = display_df.to_html(
            index=False,
            border=0
        )

        styled_html = f"""
        <div style="
            font-family:Arial,sans-serif;
            font-size:10pt;
            width:100%;
        ">

            <h3 style="
                text-align:center;
                margin-bottom:8px;
            ">
                Top Stores by Total Reported Stolen YTD
            </h3>

            <div style="
                text-align:center;
                color:gray;
                font-size:8pt;
                margin-bottom:15px;
            ">
                {store_label}
            </div>

            <div style="
                display:flex;
                justify-content:center;
                width:100%;
            ">

                {html_table}

            </div>

            <style>

                table {{
                    border-collapse: collapse;
                    min-width: 750px;
                    margin-left: auto;
                    margin-right: auto;
                }}

                th {{
                    background-color: #6da97c;
                    color: white;
                    padding: 8px;
                    text-align: center;
                }}

                td {{
                    padding: 8px;
                    text-align: center;
                    border-bottom: 1px solid #dddddd;
                }}

            </style>

        </div>
        """

        display(HTML(styled_html))

# ============================================================
# EVENTS
# ============================================================

store_selector.observe(
    build_report,
    names="value"
)

# ============================================================
# INITIAL LOAD
# ============================================================

build_report()

SelectMultiple(description='Store(s):', index=(13,), layout=Layout(height='200px', width='450px'), options=('A…

Output()

In [53]:
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, display

# ============================================================
# STORE NAME SELECTOR (MULTI-SELECT)
# ============================================================

store_options = sorted(
    df_internal_incidents["Store Name"]
    .dropna()
    .unique()
    .tolist()
)

store_selector = widgets.SelectMultiple(
    options=store_options,
    value=("Pearle Vision",)
    if "Pearle Vision" in store_options
    else (store_options[0],),
    description="Store(s):",
    rows=min(10, len(store_options)),
    layout=widgets.Layout(
        width="450px",
        height="200px"
    )
)

output = widgets.Output()

display(store_selector)
display(output)

# ============================================================
# TABLE BUILDER
# ============================================================

def generate_styled_table(
    df_source,
    group_column,
    count_column_name
):

    df_grouped = (
        df_source[group_column]
        .fillna("Blank")
        .value_counts()
        .reset_index()
    )

    df_grouped.columns = [
        group_column,
        count_column_name
    ]

    total_val = df_grouped[count_column_name].sum()

    df_grouped[count_column_name] = (
        df_grouped[count_column_name]
        .map("{:,}".format)
    )

    total_formatted = f"{total_val:,}"

    main_rows_html = ""

    for _, row in df_grouped.iterrows():

        main_rows_html += f"""
        <tr style="border-bottom:1px solid #ddd;">
            <td style="padding:10px;text-align:left;">
                {row[group_column]}
            </td>
            <td style="padding:10px;text-align:center;">
                {row[count_column_name]}
            </td>
        </tr>
        """

    total_row_html = f"""
    <tr style="
        font-weight:bold;
        border-top:2px solid black;
    ">
        <td style="
            padding:10px;
            text-align:left;
        ">
            Total
        </td>
        <td style="
            padding:10px;
            text-align:center;
        ">
            {total_formatted}
        </td>
    </tr>
    """

    return f"""
    <div style="
        font-family:Arial;
        font-size:10pt;
        margin-bottom:30px;
    ">

        <table style="
            margin:0 auto;
            border-collapse:collapse;
            width:60%;
            border:1px solid #ccc;
        ">

            <thead>

                <tr>

                    <th style="
                        background:#6da97c;
                        color:white;
                        padding:10px;
                        text-align:left;
                    ">
                        {group_column}
                    </th>

                    <th style="
                        background:#6da97c;
                        color:white;
                        padding:10px;
                        text-align:center;
                        width:30%;
                    ">
                        Count of Rows
                    </th>

                </tr>

            </thead>

            <tbody>

                {main_rows_html}

                {total_row_html}

            </tbody>

        </table>

    </div>
    """

# ============================================================
# MAIN REPORT
# ============================================================

def build_report(change=None):

    with output:

        output.clear_output()

        selected_stores = list(
            store_selector.value
        )

        if len(selected_stores) == 0:

            display(
                HTML(
                    """
                    <div style="
                        color:red;
                        font-family:Arial;
                    ">
                        Please select at least one store.
                    </div>
                    """
                )
            )

            return

        # ----------------------------------------------------
        # CLEAN HEADERS
        # ----------------------------------------------------

        df_work = df_internal_incidents.copy()

        df_work.columns = (
            df_work.columns.astype(str)
            .str.replace(
                r"\s+",
                " ",
                regex=True
            )
            .str.strip()
        )

        # ----------------------------------------------------
        # FILTER STORES
        # ----------------------------------------------------

        df_work = df_work[
            df_work["Store Name"]
            .isin(selected_stores)
        ].copy()

        # ----------------------------------------------------
        # REMOVE OPERATIONAL INVESTIGATION
        # ----------------------------------------------------

        df_work = df_work[
            df_work["Case Type"]
            != "Operational Investigation"
        ]

        if len(df_work) == 0:

            display(
                HTML(
                    """
                    <div style="
                        color:red;
                        font-family:Arial;
                    ">
                        No data found.
                    </div>
                    """
                )
            )

            return

        # ----------------------------------------------------
        # REPORT TITLE
        # ----------------------------------------------------

        if len(selected_stores) == 1:

            store_label = selected_stores[0]

        elif len(selected_stores) <= 3:

            store_label = (
                ", ".join(selected_stores)
            )

        else:

            store_label = (
                f"{len(selected_stores)} Stores Selected"
            )

        display(
            HTML(
                f"""
                <h3 style="
                    text-align:center;
                    font-family:Arial;
                    margin-bottom:5px;
                ">
                    Internal Cases / Operational Investigations
                </h3>

                <div style="
                    text-align:center;
                    color:#666;
                    font-size:9pt;
                    margin-bottom:5px;
                ">
                    {store_label}
                </div>

                <div style="
                    text-align:center;
                    color:#666;
                    font-size:8pt;
                    margin-bottom:15px;
                ">
                    Source: Internal Incidents PG
                </div>
                """
            )
        )

        # ----------------------------------------------------
        # BAR CHART
        # ----------------------------------------------------

        case_counts = (
            df_work["Case Type"]
            .fillna("Blank")
            .value_counts()
            .sort_values(
                ascending=False
            )
        )

        wrapped_labels = [
            textwrap.fill(
                str(label),
                width=15
            )
            for label in case_counts.index
        ]

        plt.figure(figsize=(10, 6))

        bars = plt.bar(
            wrapped_labels,
            case_counts.values,
            color="#6da97c"
        )

        plt.title(
            "Count of Unique Case Types "
            "(Excl. Operational Investigation)",
            fontsize=12,
            fontname="Arial",
            weight="bold",
            pad=15
        )

        plt.ylabel(
            "Count of Rows",
            fontsize=10
        )

        plt.xlabel(
            "Case Type",
            fontsize=10,
            labelpad=10
        )

        plt.xticks(
            rotation=0,
            ha="center",
            fontsize=9
        )

        for bar in bars:

            val = bar.get_height()

            plt.text(
                bar.get_x() + bar.get_width()/2,
                val + 0.1,
                f"{int(val):,}",
                ha="center",
                va="bottom",
                fontsize=9
            )

        plt.grid(
            axis="y",
            linestyle="--",
            alpha=0.5
        )

        plt.tight_layout()
        plt.show()

        # ----------------------------------------------------
        # TABLE 1
        # ----------------------------------------------------

        display(
            HTML(
                """
                <h4 style="
                    text-align:center;
                    font-family:Arial;
                ">
                    Table 1: Source Of Info
                </h4>
                """
            )
        )

        display(
            HTML(
                generate_styled_table(
                    df_work,
                    "Source Of Info",
                    "Count of Rows"
                )
            )
        )

        # ----------------------------------------------------
        # TABLE 2
        # ----------------------------------------------------

        display(
            HTML(
                """
                <h4 style="
                    text-align:center;
                    font-family:Arial;
                ">
                    Table 2: Action Taken
                </h4>
                """
            )
        )

        display(
            HTML(
                generate_styled_table(
                    df_work,
                    "Action Taken",
                    "Count of Rows"
                )
            )
        )

# ============================================================
# EVENT HANDLER
# ============================================================

store_selector.observe(
    build_report,
    names="value"
)

# ============================================================
# INITIAL LOAD
# ============================================================

build_report()

SelectMultiple(description='Store(s):', index=(7,), layout=Layout(height='200px', width='450px'), options=('Cl…

Output()